<a href="https://colab.research.google.com/github/Saifullah785/PCB_Defect_Detection_ML_Projects/blob/main/Project_02_PCB_Fault_Detection/Project_02_Preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
import xml.etree.ElementTree as ET
from shutil import copyfile
import os
import os.path as path
import shutil
import pathlib
from pathlib import Path
from tqdm import tqdm
import random

In [ ]:
FINAL_DATA_DIR = Path("./final_track_data")
clear_old_data = False
if clear_old_data and path.exists(FINAL_DATA_DIR):
    shutil.rmtree(FINAL_DATA_DIR)
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

In [ ]:

CLASSES = [
    "Short",
    "Spur",
    "Spurious copper",
    "Open",
    "Mouse bite",
    "Hole breakout",
    "Conductor scratch",
    "Conductor foreign object",
    "Base material foreign object",
    "Missing hole",
]

SHORT = 0
SPUR = 1  # Extra copper protruding out of the copper track
SPURIOUS_COPPER = 2  # Extra copper outside of a track
OPEN = 3
MOUSE_BITE = 4  # Copper removed from the track
HOLE_BREAKOUT = 5  # Hole misaligned
SCRATCH = 6
CONDUCTOR_FOREIGN_OBJECT = 7
BASE_MATERIAL_FOREIGN_OBJECT = 8
MISSING_HOLE = 9

In [ ]:
DATASET_GROUPS = ["train", "test", "valid"]
for g in DATASET_GROUPS:
    os.mkdir(FINAL_DATA_DIR / g)

In [ ]:
with open(FINAL_DATA_DIR / "data.yaml", "w") as f:
    f.write(
        f"""train: ../train/images
val: ../valid/images
test: ../test/images

nc: {len(CLASSES)}
names: [{",".join(f"'{s}'" for s in CLASSES)}]
"""
    )

In [ ]:
def map_one_dataset_group(
    from_path_top: Path, to_path_top: Path, dataset_group: str, mapping: dict[int, int]
):
    from_path = from_path_top / dataset_group
    to_path = to_path_top / dataset_group
    shutil.copytree(from_path / "images", to_path / "images", dirs_exist_ok=True)
    to_labels_path = to_path / "labels"
    to_labels_path.mkdir(parents=True, exist_ok=True)
    for from_child_path in tqdm((from_path / "labels").iterdir()):
        to_child_path = to_labels_path / from_child_path.name
        with open(from_child_path, "r") as f:
            with open(to_child_path, "w") as t:
                for line in f:
                    class_id, x, y, w, h = line.strip().split()
                    new_class_id = str(mapping[int(class_id)])
                    t.write(f"{new_class_id} {x} {y} {w} {h}\n")


def map_dataset(from_path_top: Path, to_path_top: Path, mapping: dict[int, int]):
    for g in tqdm(DATASET_GROUPS):
        map_one_dataset_group(from_path_top, to_path_top, g, mapping)

In [ ]:
def convert_bbox_to_yolo(
    size: tuple[int, int], box: tuple[int, int, int, int]
) -> tuple[int, int, int, int]:
    """Convert bounding box coordinates from PASCAL VOC format to YOLO format.

    :param size: A tuple of the image size: (width, height)
    :param box: A tuple of the PASCAL VOC bbox: (xmin, ymin, xmax, ymax)
    :return: A tuple of the YOLO bbox: (x_center, y_center, width, height)
    """
    # Calculate relative dimensions
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]

    # Calculate center, width, and height of the bbox in relative dimension
    rel_x_center = (box[0] + box[2]) / 2.0 * dw
    rel_y_center = (box[1] + box[3]) / 2.0 * dh
    rel_width = (box[2] - box[0]) * dw
    rel_height = (box[3] - box[1]) * dh

    return (rel_x_center, rel_y_center, rel_width, rel_height)

In [ ]:
def xml_to_txt(input_file: Path, output_txt: Path, classes: dict[str, int]):
    """Parse an XML file in PASCAL VOC format and convert it to YOLO format.

    :param input_xml: Path to the input XML file.
    :param output_txt: Path to the output .txt file in YOLO format.
    :param classes: A list of class names as strings.
    """
    # Load and parse the XML file
    if input_file.suffix == ".txt":
        # Try to parse .txt as XML
        try:
            # Attempt to parse the file content as XML
            with input_file.open("r", encoding="utf-8") as file:
                file_content = file.read()
            root = ET.fromstring(file_content)
        except ET.ParseError as e:
            print(f"Error parsing {input_file}: {e}")
            return  # Skip this file and continue with the next
    else:
        # Try parsing the XML file (expects XML format)
        try:
            tree = ET.parse(input_file)
            root = tree.getroot()
        except ET.ParseError as e:
            print(f"Error parsing {input_file}: {e}")
            return  # Skip this file and continue with the next

    # Extract image dimensions
    size_element = root.find("size")
    image_width = int(size_element.find("width").text)
    image_height = int(size_element.find("height").text)

    with output_txt.open("w") as file:
        # Process each object in the XML
        for obj in root.iter("object"):
            is_difficult = obj.find("difficult").text
            class_name = obj.find("name").text

            # Skip "difficult" objects or if the name is not in classes
            if class_name not in classes or int(is_difficult) == 1:
                continue

            class_id = classes[class_name]

            # Extract and convert bbox
            xml_box = obj.find("bndbox")
            bbox = (
                float(xml_box.find("xmin").text),
                float(xml_box.find("ymin").text),
                float(xml_box.find("xmax").text),
                float(xml_box.find("ymax").text),
            )
            yolo_bbox = convert_bbox_to_yolo((image_width, image_height), bbox)

            # Write to the output file in YOLO format
            file.write(f"{class_id} {' '.join(map(str, yolo_bbox))}\n")

In [ ]:
def map_pascal_to_yolo_one_dataset_group(
    files_list: list[tuple[Path, Path]],
    to_path_with_group: Path,
    mapping: dict[str, int],
):
    to_images_path = to_path_with_group / "images"
    to_labels_path = to_path_with_group / "labels"
    to_images_path.mkdir(parents=True, exist_ok=True)
    to_labels_path.mkdir(parents=True, exist_ok=True)
    for image, xml in tqdm(files_list):
        shutil.copy(image, to_images_path / image.name)
        to_child_path = to_labels_path / xml.with_suffix(".txt").name
        xml_to_txt(xml, to_child_path, mapping)


def map_pascal_to_yolo_dataset(
    files_list: list[tuple[Path, Path]], to_path_top: Path, mapping: dict[int, int]
):
    rng = random.Random(x=42)
    rng.shuffle(files_list)
    train_end_idx = int(len(files_list) * 0.7)
    test_end_idx = int(len(files_list) * (0.7 + 0.15))
    train_files = files_list[0:train_end_idx]
    test_files = files_list[train_end_idx:test_end_idx]
    val_files = files_list[test_end_idx:]
    vs = [train_files, test_files, val_files]
    for i, g in tqdm(enumerate(DATASET_GROUPS)):
        map_pascal_to_yolo_one_dataset_group(vs[i], to_path_top / g, mapping)

In [ ]:
MIXED_PCB_DEFECT_DATASET_MAPPING = {
    0: MISSING_HOLE,
    1: MOUSE_BITE,
    2: OPEN,
    3: SHORT,
    4: SPUR,
    5: SPURIOUS_COPPER,
}

In [ ]:
# map_dataset(
#     Path("./MIXED PCB DEFECT DETECTION/"),
#     FINAL_DATA_DIR,
#     MIXED_PCB_DEFECT_DATASET_MAPPING,
# )